# 대청호 녹조 예측 AI 모델 - 데이터 전처리
## 1. 데이터 통합 (수질/조류 + 기상)


In [7]:
import pandas as pd
import os

# 1. 파일 경로 설정 (폴더 구조에 맞게)
WATER_PATH = './data/water_quality/'
WEATHER_PATH = './data/weather/'

print("데이터 통합을 시작합니다. 얍! 🪄")

# =========================================================
# [STEP 1] 수질(녹조) 데이터 통합 (환경부 엑셀)
# =========================================================
water_list = []
for f in os.listdir(WATER_PATH):
    if f.endswith('.xlsx') or f.endswith('.xls'):
        engine = 'openpyxl' if f.endswith('.xlsx') else 'xlrd'
        
        # header=1: 첫 번째 껍데기 제목 줄은 무시하고 두 번째 줄부터 진짜 제목으로 읽음
        temp_df = pd.read_excel(os.path.join(WATER_PATH, f), engine=engine, header=1)
        water_list.append(temp_df)

df_water = pd.concat(water_list, ignore_index=True)

# 🧹 수질 데이터 청소: 중간중간 끼어있는 '가짜 제목 줄'과 빈칸 싹 날려버리기
df_water = df_water[df_water['지점명'] != '지점명']
df_water = df_water[df_water['분류'] != '분류']
df_water = df_water.dropna(subset=['조사일']) # 날짜가 아예 없는 유령 데이터 삭제
df_water = df_water.reset_index(drop=True)    # 줄 번호 0번부터 다시 예쁘게 정렬

print(f"✅ 수질 데이터 통합 및 청소 완료! (크기: {df_water.shape})")

# =========================================================
# [STEP 2] 기상 데이터 통합 (기상청 꼼수 돌파)
# =========================================================
weather_list = []
for f in os.listdir(WEATHER_PATH):
    if f.endswith('.xls') or f.endswith('.xlsx') or f.endswith('.csv'):
        file_path = os.path.join(WEATHER_PATH, f)
        try:
            # 1차 시도: 진짜 엑셀인지 확인
            engine = 'xlrd' if f.endswith('.xls') else 'openpyxl'
            temp_df = pd.read_excel(file_path, engine=engine)
        except Exception:
            # 2차 시도: 엑셀인 척하는 텍스트 파일일 경우 (탭 기준)
            try:
                temp_df = pd.read_csv(file_path, sep='\t', encoding='cp949')
            except:
                # 3차 시도: 쉼표 기준일 경우 대비
                temp_df = pd.read_csv(file_path, sep=',', encoding='cp949')
                
        weather_list.append(temp_df)

df_weather = pd.concat(weather_list, ignore_index=True)
print(f"✅ 기상 데이터 통합 완료! (크기: {df_weather.shape})")

# =========================================================
# [STEP 3] 결과 확인
# =========================================================
print("\n--- 💧 수질 데이터 상위 5개 ---")
display(df_water.head(5))

print("\n--- ☀️ 기상 데이터 상위 5개 ---")
display(df_weather.head(5))

데이터 통합을 시작합니다. 얍! 🪄
✅ 수질 데이터 통합 및 청소 완료! (크기: (1828, 23))
✅ 기상 데이터 통합 완료! (크기: (4134, 7))

--- 💧 수질 데이터 상위 5개 ---


,분류,지점명,채수위치,조사일,수온(℃),pH,DO(㎎/L),투명도,탁도,Chl-a (㎎/㎥),...,유해남조류 우점종 및 세포수(속별).2,유해남조류 우점종 및 세포수(속별).3,냄새물질 (ng/L),냄새물질 (ng/L).1,조류독소(㎍/L),조류독소(㎍/L).1,조류독소(㎍/L).2,조류독소(㎍/L).3,조류독소(㎍/L).4,조류독소(㎍/L).5
0,호소,대청호,문의,2015.01.05,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,호소,대청호,문의,2015.01.12,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,호소,대청호,문의,2015.01.19,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,호소,대청호,문의,2015.01.26,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,호소,대청호,문의,2015.02.02,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- ☀️ 기상 데이터 상위 5개 ---


,지점,지점명,일시,평균기온(°C),일강수량(mm),합계 일조시간(hr),합계 일사량(MJ/m2)
0,133,대전,2015-01-01,-5.6,0.0,8.0,10.53
1,133,대전,2015-01-02,-4.7,0.0,5.4,8.42
2,133,대전,2015-01-03,-2.9,NaN,8.6,11.35
3,133,대전,2015-01-04,2.0,NaN,5.0,8.28
4,133,대전,2015-01-05,3.2,6.1,4.1,7.64


In [8]:
import pandas as pd
import os

# 1. 파일 경로 설정
WATER_PATH = './data/water_quality/'
WEATHER_PATH = './data/weather/'

print("데이터 통합 및 병합을 시작합니다. 얍! 🪄")

# =========================================================
# [STEP 1] 수질(녹조) 데이터 통합 및 청소
# =========================================================
water_list = []
for f in os.listdir(WATER_PATH):
    if f.endswith(('.xlsx', '.xls')):
        engine = 'openpyxl' if f.endswith('.xlsx') else 'xlrd'
        temp_df = pd.read_excel(os.path.join(WATER_PATH, f), engine=engine, header=1)
        water_list.append(temp_df)

df_water = pd.concat(water_list, ignore_index=True)

# 가짜 제목 줄 및 빈칸 청소
df_water = df_water[df_water['지점명'] != '지점명']
df_water = df_water[df_water['분류'] != '분류']
df_water = df_water.dropna(subset=['조사일'])
df_water = df_water.reset_index(drop=True)

# =========================================================
# [STEP 2] 기상 데이터 통합 (기상청 꼼수 돌파)
# =========================================================
weather_list = []
for f in os.listdir(WEATHER_PATH):
    if f.endswith(('.xls', '.xlsx', '.csv')):
        file_path = os.path.join(WEATHER_PATH, f)
        try:
            engine = 'xlrd' if f.endswith('.xls') else 'openpyxl'
            temp_df = pd.read_excel(file_path, engine=engine)
        except Exception:
            try:
                temp_df = pd.read_csv(file_path, sep='\t', encoding='cp949')
            except:
                temp_df = pd.read_csv(file_path, sep=',', encoding='cp949')
                
        weather_list.append(temp_df)

df_weather = pd.concat(weather_list, ignore_index=True)

# =========================================================
# 🚀 [STEP 3] 날짜 형식 통일 및 데이터 병합 (Merge)
# =========================================================
# 1. 제각각인 날짜를 판다스 공식 날짜 형식으로 변환
df_water['조사일'] = pd.to_datetime(df_water['조사일'])
df_weather['일시'] = pd.to_datetime(df_weather['일시'])

# 2. 날짜 과거순으로 예쁘게 정렬
df_water = df_water.sort_values(by='조사일').reset_index(drop=True)
df_weather = df_weather.sort_values(by='일시').reset_index(drop=True)

# 3. 데이터 합체! (수질 데이터가 있는 날짜를 기준으로 기상 데이터를 옆에 갖다 붙임)
df_master = pd.merge(df_water, df_weather, left_on='조사일', right_on='일시', how='left')

print(f"✅ 최종 마스터 데이터셋 병합 완료! (총 {df_master.shape[1]}개 항목)")

# 4. 합쳐진 마스터 데이터 확인
display(df_master.head(5))

데이터 통합 및 병합을 시작합니다. 얍! 🪄
✅ 최종 마스터 데이터셋 병합 완료! (총 30개 항목)


,분류,지점명_x,채수위치,조사일,수온(℃),pH,DO(㎎/L),투명도,탁도,Chl-a (㎎/㎥),...,조류독소(㎍/L).3,조류독소(㎍/L).4,조류독소(㎍/L).5,지점,지점명_y,일시,평균기온(°C),일강수량(mm),합계 일조시간(hr),합계 일사량(MJ/m2)
0,호소,대청호,문의,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,133.0,대전,2015-01-05,3.2,6.1,4.1,7.64
1,호소,대청호,회남,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,133.0,대전,2015-01-05,3.2,6.1,4.1,7.64
2,호소,대청호,추동,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,133.0,대전,2015-01-05,3.2,6.1,4.1,7.64
3,호소,대청호,추동,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,133.0,대전,2015-01-12,-1.8,NaN,8.9,12.35
4,호소,대청호,문의,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,133.0,대전,2015-01-12,-1.8,NaN,8.9,12.35


In [10]:
import pandas as pd
import os

WATER_PATH = './data/water_quality/'
WEATHER_PATH = './data/weather/'

print("데이터 통합 및 병합을 시작합니다. 얍! 🪄")

# =========================================================
# [STEP 1] 수질(녹조) 데이터 통합 및 청소
# =========================================================
water_list = []
for f in os.listdir(WATER_PATH):
    if f.endswith(('.xlsx', '.xls')):
        engine = 'openpyxl' if f.endswith('.xlsx') else 'xlrd'
        temp_df = pd.read_excel(os.path.join(WATER_PATH, f), engine=engine, header=1)
        water_list.append(temp_df)

df_water = pd.concat(water_list, ignore_index=True)

df_water = df_water[df_water['지점명'] != '지점명']
df_water = df_water[df_water['분류'] != '분류']
df_water = df_water.dropna(subset=['조사일'])
df_water = df_water.reset_index(drop=True)

# =========================================================
# [STEP 2] 기상 데이터 통합 (기상청 꼼수 돌파)
# =========================================================
weather_list = []
for f in os.listdir(WEATHER_PATH):
    if f.endswith(('.xls', '.xlsx', '.csv')):
        file_path = os.path.join(WEATHER_PATH, f)
        try:
            engine = 'xlrd' if f.endswith('.xls') else 'openpyxl'
            temp_df = pd.read_excel(file_path, engine=engine)
        except Exception:
            try:
                temp_df = pd.read_csv(file_path, sep='\t', encoding='cp949')
            except:
                temp_df = pd.read_csv(file_path, sep=',', encoding='cp949')
                
        weather_list.append(temp_df)

df_weather = pd.concat(weather_list, ignore_index=True)

# =========================================================
# 🚀 [STEP 3] 날짜 형식 통일 및 데이터 병합 (Merge)
# =========================================================
df_water['조사일'] = pd.to_datetime(df_water['조사일'])
df_weather['일시'] = pd.to_datetime(df_weather['일시'])

df_water = df_water.sort_values(by='조사일').reset_index(drop=True)
df_weather = df_weather.sort_values(by='일시').reset_index(drop=True)

df_master = pd.merge(df_water, df_weather, left_on='조사일', right_on='일시', how='left')

# =========================================================
# ✨ [STEP 4] 중복 컬럼 청소 (깔끔하게 다듬기)
# =========================================================
# 불필요해진 기상 지점 정보와 중복 날짜 삭제
df_master = df_master.drop(columns=['지점', '지점명_y', '일시'])
# 찌그러진 지점명_x 를 원래 이름인 '지점명'으로 복구
df_master = df_master.rename(columns={'지점명_x': '지점명'})

print(f"✅ 최종 마스터 데이터셋 병합 완료! (총 {df_master.shape[1]}개 항목)")
display(df_master.head(5))

# =========================================================
# ✨ [STEP 5] 결측치(NaN) 과학적으로 채우기
# =========================================================
print("비어있는 데이터를 채웁니다...")

# 1. 강수량 빈칸은 '비가 안 옴'이므로 0으로 채우기
if '일강수량(mm)' in df_master.columns:
    df_master['일강수량(mm)'] = df_master['일강수량(mm)'].fillna(0)

# 2. 숫자형 데이터만 골라서 '선형 보간법'으로 자연스럽게 이어주기
numeric_cols = df_master.select_dtypes(include=['float64', 'int64']).columns
df_master[numeric_cols] = df_master[numeric_cols].interpolate(method='linear')

# 3. 맨 앞이나 맨 뒤에 있어서 선형 보간이 안 닿은 빈칸들은 앞뒤 값으로 마저 채우기
df_master[numeric_cols] = df_master[numeric_cols].bfill().ffill()

print("✅ 결측치 처리 완료!")

# =========================================================
# 💾 [STEP 6] 최종 마스터 데이터셋 파일로 저장하기
# =========================================================
# data 폴더 안에 csv 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
save_path = './data/대청호_최종_마스터데이터.csv'
df_master.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"🎉 파일 저장 성공! (저장 위치: {save_path})")

# 최종 완성된 데이터 확인
display(df_master.head(5))

데이터 통합 및 병합을 시작합니다. 얍! 🪄
✅ 최종 마스터 데이터셋 병합 완료! (총 27개 항목)


,분류,지점명,채수위치,조사일,수온(℃),pH,DO(㎎/L),투명도,탁도,Chl-a (㎎/㎥),...,조류독소(㎍/L),조류독소(㎍/L).1,조류독소(㎍/L).2,조류독소(㎍/L).3,조류독소(㎍/L).4,조류독소(㎍/L).5,평균기온(°C),일강수량(mm),합계 일조시간(hr),합계 일사량(MJ/m2)
0,호소,대청호,문의,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
1,호소,대청호,회남,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
2,호소,대청호,추동,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
3,호소,대청호,추동,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.8,NaN,8.9,12.35
4,호소,대청호,문의,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.8,NaN,8.9,12.35


비어있는 데이터를 채웁니다...
✅ 결측치 처리 완료!
🎉 파일 저장 성공! (저장 위치: ./data/대청호_최종_마스터데이터.csv)


,분류,지점명,채수위치,조사일,수온(℃),pH,DO(㎎/L),투명도,탁도,Chl-a (㎎/㎥),...,조류독소(㎍/L),조류독소(㎍/L).1,조류독소(㎍/L).2,조류독소(㎍/L).3,조류독소(㎍/L).4,조류독소(㎍/L).5,평균기온(°C),일강수량(mm),합계 일조시간(hr),합계 일사량(MJ/m2)
0,호소,대청호,문의,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
1,호소,대청호,회남,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
2,호소,대청호,추동,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.2,6.1,4.1,7.64
3,호소,대청호,추동,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.8,0.0,8.9,12.35
4,호소,대청호,문의,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.8,0.0,8.9,12.35


In [12]:
import pandas as pd
import os

# 1. 파일 경로 설정
WATER_PATH = './data/water_quality/'
WEATHER_PATH = './data/weather/'

print("🚀 마지막 데이터 대청소를 시작합니다!")

# =========================================================
# [STEP 1] 수질(녹조) 데이터 통합 및 청소
# =========================================================
water_list = []
for f in os.listdir(WATER_PATH):
    if f.endswith(('.xlsx', '.xls')):
        engine = 'openpyxl' if f.endswith('.xlsx') else 'xlrd'
        temp_df = pd.read_excel(os.path.join(WATER_PATH, f), engine=engine, header=1)
        water_list.append(temp_df)

df_water = pd.concat(water_list, ignore_index=True)

# 🧹 1차 청소: 가짜 제목 줄 및 유령 데이터 제거
df_water = df_water[df_water['지점명'] != '지점명']
df_water = df_water[df_water['분류'] != '분류']
df_water = df_water.dropna(subset=['조사일'])

# =========================================================
# [STEP 2] 기상 데이터 통합 (기상청 꼼수 돌파)
# =========================================================
weather_list = []
for f in os.listdir(WEATHER_PATH):
    if f.endswith(('.xls', '.xlsx', '.csv')):
        file_path = os.path.join(WEATHER_PATH, f)
        try:
            engine = 'xlrd' if f.endswith('.xls') else 'openpyxl'
            temp_df = pd.read_excel(file_path, engine=engine)
        except Exception:
            try:
                temp_df = pd.read_csv(file_path, sep='\t', encoding='cp949')
            except:
                temp_df = pd.read_csv(file_path, sep=',', encoding='cp949')
        weather_list.append(temp_df)

df_weather = pd.concat(weather_list, ignore_index=True)

# =========================================================
# [STEP 3] 날짜 통일 및 데이터 병합 (Merge)
# =========================================================
df_water['조사일'] = pd.to_datetime(df_water['조사일'])
df_weather['일시'] = pd.to_datetime(df_weather['일시'])

df_water = df_water.sort_values(by='조사일').reset_index(drop=True)
df_weather = df_weather.sort_values(by='일시').reset_index(drop=True)

df_master = pd.merge(df_water, df_weather, left_on='조사일', right_on='일시', how='left')

# 중복 컬럼 정리
df_master = df_master.drop(columns=['지점', '지점명_y', '일시'])
df_master = df_master.rename(columns={'지점명_x': '지점명'})

# =========================================================
# ✨ [STEP 4] 글자 데이터(정량한계미만)를 숫자 0으로 변환
# =========================================================
print("데이터 속 한글(정량한계미만 등)을 숫자로 바꾸는 중...")

# 1. '정량한계미만'이나 기타 글자들을 0으로 치환
df_master = df_master.replace('정량한계미만', 0)
df_master = df_master.replace('결측', 0) # 혹시 모를 '결측' 글자 대비

# 2. 모든 숫자 컬럼들을 진짜 '숫자 타입'으로 강제 변환 (글자가 섞여있어서 문자로 인식될 수 있음)
for col in df_master.columns:
    if col not in ['분류', '지점명', '채수위치', '조사일', '유해남조류 우점종 및 세포수(속별).2', '유해남조류 우점종 및 세포수(속별).3']:
        df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# =========================================================
# 🪄 [STEP 5] 결측치(NaN) 과학적으로 채우기
# =========================================================
if '일강수량(mm)' in df_master.columns:
    df_master['일강수량(mm)'] = df_master['일강수량(mm)'].fillna(0)

# 숫자형 데이터 선형 보간 (자연스럽게 잇기)
numeric_cols = df_master.select_dtypes(include=['float64', 'int64']).columns
df_master[numeric_cols] = df_master[numeric_cols].interpolate(method='linear')
df_master[numeric_cols] = df_master[numeric_cols].bfill().ffill()

print("✅ 모든 결측치 및 글자 데이터 처리 완료!")

# =========================================================
# 💾 [STEP 6] 최종 파일 저장
# =========================================================
save_path = './data/대청호_최종_마스터데이터.csv'
df_master.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"🎉 축하합니다! 완벽한 데이터가 저장되었습니다: {save_path}")
display(df_master.head(10))

🚀 마지막 데이터 대청소를 시작합니다!
데이터 속 한글(정량한계미만 등)을 숫자로 바꾸는 중...
✅ 모든 결측치 및 글자 데이터 처리 완료!
🎉 축하합니다! 완벽한 데이터가 저장되었습니다: ./data/대청호_최종_마스터데이터.csv


C:\Users\pc\AppData\Local\Temp\ipykernel_15492\2459501694.py:67: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_master = df_master.replace('정량한계미만', 0)


,분류,지점명,채수위치,조사일,수온(℃),pH,DO(㎎/L),투명도,탁도,Chl-a (㎎/㎥),...,조류독소(㎍/L),조류독소(㎍/L).1,조류독소(㎍/L).2,조류독소(㎍/L).3,조류독소(㎍/L).4,조류독소(㎍/L).5,평균기온(°C),일강수량(mm),합계 일조시간(hr),합계 일사량(MJ/m2)
0,호소,대청호,문의,2015-01-05,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,3.2,6.1,4.1,7.64
1,호소,대청호,회남,2015-01-05,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,3.2,6.1,4.1,7.64
2,호소,대청호,추동,2015-01-05,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,3.2,6.1,4.1,7.64
3,호소,대청호,추동,2015-01-12,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,-1.8,0.0,8.9,12.35
4,호소,대청호,문의,2015-01-12,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,-1.8,0.0,8.9,12.35
5,호소,대청호,회남,2015-01-12,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,-1.8,0.0,8.9,12.35
6,호소,대청호,회남,2015-01-19,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,1.8,0.0,2.3,5.16
7,호소,대청호,문의,2015-01-19,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,1.8,0.0,2.3,5.16
8,호소,대청호,추동,2015-01-19,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,1.8,0.0,2.3,5.16
9,호소,대청호,회남,2015-01-26,8.8,8.2,13.3,2.9,1.7,4.5,...,0.0,0.7,0.0,0.0,0.0,0.0,5.3,0.3,0.0,1.82
